# How an ontology drives extraction — and how to extend yours

Most LLM extraction notebooks treat the prompt as a black box. SEOCHO doesn't — the prompt is *generated from your ontology*. This notebook makes that visible:

1. Print the **live extraction prompt** seocho synthesizes from the active ontology so you can see exactly what the LLM is asked to do.
2. Index the **same FinDER document twice** — once with a stripped-down `Entity / RELATED_TO` baseline, once with FIBO. Compare what comes out the other side.
3. **DIY:** add your own class to the ontology and re-index — see the new label show up *and* see how the prompt changed.

Backend is the bundled DozerDB; we use one workspace per experiment so the runs stay isolated. Open the **Neo4j Browser** at http://localhost:7474 to follow along.

## Setup

In [ ]:
import json
import os
import sys
import time
from copy import deepcopy
from pathlib import Path

from dotenv import load_dotenv

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "seocho").is_dir() and (p / "examples").is_dir():
            return p
        p = p.parent
    return Path.cwd()

ROOT = _find_repo_root()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

LLM_SPEC = os.environ.get("SEOCHO_LLM", "openai/gpt-4o-mini")
LLM_PROVIDER, LLM_MODEL = (LLM_SPEC.split("/", 1) + [""])[:2]
if not LLM_MODEL:
    LLM_PROVIDER, LLM_MODEL = "openai", LLM_SPEC

PROVIDER_KEY = {
    "openai": "OPENAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "kimi": "MOONSHOT_API_KEY",
    "grok": "XAI_API_KEY",
    "qwen": "DASHSCOPE_API_KEY",
}.get(LLM_PROVIDER, "OPENAI_API_KEY")
if not os.environ.get(PROVIDER_KEY):
    raise RuntimeError(f"{PROVIDER_KEY} required for SEOCHO_LLM={LLM_SPEC}")

NEO4J_URI      = os.environ.get("NEO4J_URI", "bolt://tutorials-neo4j:7687")
NEO4J_USER     = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "tutorialspw")

FINDER_PATH = os.environ.get(
    "FINDER_PATH",
    str(ROOT / "examples" / "finder" / "datasets" / "finder_tutorial_subset.json"),
)
WORKDIR = ROOT / ".seocho" / "finder_ontology_drives"
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"FinDER source: {FINDER_PATH}")
print(f"LLM:           {LLM_PROVIDER}/{LLM_MODEL}")
print(f"Graph backend: Neo4j @ {NEO4J_URI}")

In [ ]:
from seocho.benchmarking import load_finder_cases

# We pick *one* FinDER doc to build a clear before/after story. Swap
# DEMO_INDEX to a different number to try other documents.
cases = load_finder_cases(FINDER_PATH)
DEMO_INDEX = 4   # Meta Platforms / FTC litigation — has lawsuits, settlements, regulators
demo = cases[DEMO_INDEX]
print(f"[{demo.category}] {demo.case_id}")
print(f"Q: {demo.question}")
print(f"\nText:\n{demo.text}")

## 1. Inspect the active ontology

Before we look at the prompt, let's see what the ontology actually declares. SEOCHO ontologies are JSON-LD on disk; the in-memory `Ontology` object exposes node and relationship definitions you can iterate.

In [ ]:
from seocho import Ontology

fibo = Ontology.from_jsonld(ROOT / "examples" / "datasets" / "fibo_base.jsonld")

print(f"Ontology: {fibo.name} (v{fibo.version})")
print(f"\n{'Class':18s} {'Aliases':30s} Description")
print("-" * 80)
for label, node in fibo.nodes.items():
    aliases = ", ".join(node.aliases or []) or "—"
    print(f"{label:18s} {aliases:30s} {node.description or '—'}")

print(f"\n{'Relationship':18s} {'Source -> Target':30s} Description")
print("-" * 80)
for rtype, rel in fibo.relationships.items():
    arrow = f"{rel.source} -> {rel.target}"
    print(f"{rtype:18s} {arrow:30s} {rel.description or '—'}")

## 2. The prompt the LLM sees

This is the lesson. SEOCHO compiles the ontology into a prompt; the prompt is what the LLM is *actually instructed with*. Read it carefully — every label, every relationship, every alias from the ontology shows up here. Change the ontology, the prompt changes.

In [ ]:
from seocho.query.strategy import ExtractionStrategy

def render_prompt(ontology: Ontology, sample_text: str) -> str:
    """Return the exact extraction prompt seocho will hand the LLM.

    seocho's extraction prompt has two parts: a *system* message with
    the ontology contract, and a *user* message with the document text.
    We concatenate them with section markers so the diff later is
    easy to read.
    """
    strategy = ExtractionStrategy(ontology)
    system, user = strategy.render(sample_text)
    return f"=== SYSTEM ===\n{system}\n\n=== USER ===\n{user}"

fibo_prompt = render_prompt(fibo, demo.text)
print("=" * 80)
print("PROMPT THE LLM SEES (FIBO base ontology)")
print("=" * 80)
print(fibo_prompt[:3500])
print("…" if len(fibo_prompt) > 3500 else "")

## 3. Same document, two ontologies, two graphs

Now we extract from the same FinDER document twice:

- **Generic baseline** — single `Entity` class, single `RELATED_TO` relationship. The LLM has no schema to lean on.
- **FIBO base** — Company / Person / FinancialMetric / Regulation with typed relationships.

Same text, same model, same temperature. The only thing that changes is the ontology — i.e., the prompt.

In [ ]:
from seocho import Seocho
from seocho.index.pipeline import IndexingPipeline
from seocho.store.graph import Neo4jGraphStore
from seocho.store.llm import create_llm_backend

# Generic baseline ontology
generic = Ontology.from_dict({
    "graph_type": "generic",
    "version": "1.0.0",
    "description": "Generic baseline — no domain schema",
    "graph_model": "lpg",
    "nodes": {"Entity": {"description": "Any named thing", "properties": {
        "name": {"type": "STRING", "constraint": "UNIQUE"}}}},
    "relationships": {"RELATED_TO": {"source": "Entity", "target": "Entity",
        "description": "Generic association"}},
})

graph_store = Neo4jGraphStore(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
llm = create_llm_backend(provider=LLM_PROVIDER, model=LLM_MODEL)

captures = {}   # ontology_label -> {"nodes": [...], "rels": [...]}

def index_with_ontology(label: str, ontology: Ontology):
    workspace = f"ontology_demo_{label}"
    graph_store.execute_write(
        "MATCH (n) WHERE n._workspace_id = $w DETACH DELETE n",
        params={"w": workspace},
    )
    captured = {"nodes": [], "rels": []}
    def cap(nodes, rels):
        captured["nodes"].extend(nodes)
        captured["rels"].extend(rels)
        return nodes, rels
    pipeline = IndexingPipeline(
        ontology=ontology,
        graph_store=graph_store,
        llm=llm,
        workspace_id=workspace,
        on_after_extract=cap,
    )
    pipeline.index(demo.text, database="neo4j", metadata={"case_id": demo.case_id})
    captured["workspace"] = workspace
    captures[label] = captured
    print(f"[{label}] {len(captured['nodes'])} nodes, {len(captured['rels'])} rels")
    print(f"  labels:    {sorted({n.get('label') for n in captured['nodes']})}")
    print(f"  rel types: {sorted({r.get('type') for r in captured['rels']})}")
    return captured

index_with_ontology("generic", generic)
index_with_ontology("fibo", fibo)

In [ ]:
from examples.finder.lib.graph_viz import draw_lpg
import matplotlib.pyplot as plt

for key, title in (("generic", "Generic baseline"), ("fibo", "FIBO base")):
    cap = captures[key]
    fig = draw_lpg(
        cap["nodes"], cap["rels"],
        title=f"{title}  —  {len(cap['nodes'])} nodes / {len(cap['rels'])} rels",
        figsize=(11, 6),
    )
    plt.show()

**What you should see:**

- *Generic*: a flat blob of `Entity` nodes connected by `RELATED_TO`. The LLM extracted *something*, but every node is the same type and every edge means the same thing. That's not a knowledge graph — it's an unlabeled bag of mentions.
- *FIBO*: typed nodes (Company / Person / FinancialMetric / Regulation) with meaningful edges (EMPLOYS / FOLLOWS / REPORTED). The same text, same model, same temperature — only the ontology changed.

The ontology was never a passive declaration. It was *the prompt the LLM saw*.

## 4. DIY: extend the ontology with your own class

FIBO base doesn't model lawsuits — but our demo document is full of them. Let's add a `Lawsuit` class plus a `DEFENDANT_IN` relationship, re-index, and watch the new label appear.

Two ways to add a class:

1. **In code** — mutate the in-memory `Ontology` object directly (what we do here).
2. **As TTL** — write a short TTL file and `+` it onto the base via `Ontology.from_ttl(...)` (Tutorial 4 demonstrates this approach).

Either way, what matters is the prompt the LLM sees changes.

In [ ]:
from seocho.ontology import NodeDef, P, PropertyType, RelDef

# Build a *small* ontology that contains only the new pieces, then add
# it to FIBO with the `+` operator. This is the canonical way to
# extend: it returns a fresh Ontology object, avoids mutating the
# original, and clears the prompt-render cache automatically.
#
# Compare to direct dict-mutation (`fibo.nodes["Lawsuit"] = ...`),
# which Ontology supports but caches stale prompt strings on — the
# new class wouldn't show up in the diff cell below.
lawsuit_overlay = Ontology(
    name="lawsuit_overlay",
    nodes={
        "Lawsuit": NodeDef(
            description="A legal proceeding the entity is party to",
            aliases=["Litigation", "Action", "Settlement"],
            properties={
                "name": P(type=PropertyType.STRING, unique=True),
                "counterparty": P(type=PropertyType.STRING),
                "status": P(type=PropertyType.STRING),
            },
        ),
    },
    relationships={
        "DEFENDANT_IN": RelDef(
            source="Company",
            target="Lawsuit",
            description="Entity is the defendant in a lawsuit",
            cardinality="ONE_TO_MANY",
        ),
    },
)

fibo_plus_lawsuit = fibo + lawsuit_overlay   # Ontology.__add__ → merge(union)
fibo_plus_lawsuit.name = "fibo_plus_lawsuit"

print(f"Composed ontology has {len(fibo_plus_lawsuit.nodes)} classes, "
      f"{len(fibo_plus_lawsuit.relationships)} relationship types.")
print("New class added: Lawsuit (aliases: Litigation, Action, Settlement)")
print("New rel added:   Company -[DEFENDANT_IN]-> Lawsuit")

In [ ]:
index_with_ontology("fibo_plus_lawsuit", fibo_plus_lawsuit)

In [ ]:
from examples.finder.lib.graph_viz import draw_lpg
import matplotlib.pyplot as plt

cap = captures["fibo_plus_lawsuit"]
fig = draw_lpg(
    cap["nodes"], cap["rels"],
    title=f"FIBO + Lawsuit  —  {len(cap['nodes'])} nodes / {len(cap['rels'])} rels",
    figsize=(11, 6),
)
plt.show()

# Did the new label show up?
lawsuit_nodes = [n for n in cap["nodes"] if n.get("label") == "Lawsuit"]
print(f"\nLawsuit nodes extracted: {len(lawsuit_nodes)}")
for n in lawsuit_nodes:
    props = n.get("properties", {})
    print(f"  {props.get('name', '?')}")

## 5. What changed in the prompt

This is the closing of the loop: the **only thing different** between the FIBO run and the FIBO-plus-Lawsuit run is the prompt. Diff them line by line:

In [ ]:
import difflib

before = render_prompt(fibo, demo.text).splitlines(keepends=True)
after  = render_prompt(fibo_plus_lawsuit, demo.text).splitlines(keepends=True)

# Show only added / removed lines (skip context for compactness)
diff = list(difflib.unified_diff(before, after, fromfile="fibo", tofile="fibo+Lawsuit", n=1))
for line in diff[:60]:
    print(line, end="")

## What to take away

- **The ontology is the prompt.** When you add or remove a class, you're literally rewriting what the LLM is asked to extract. Nothing magical happens between your YAML/JSON-LD/TTL file and the LLM call.
- **Generic schemas produce generic graphs.** Every additional class you declare gives the LLM somewhere typed to put a fact instead of dumping everything into `Entity`.
- **Aliases matter.** The `Lawsuit` class lists `Litigation`, `Action`, `Settlement` as aliases — those words appear in the document but `Lawsuit` doesn't. The aliases are what let the LLM map the surface text to your canonical label.
- **Iteration is cheap.** Add a class, re-extract, look at the graph. If you don't like what you got, change the description / aliases and try again. There's no compilation step.

Three workspaces are now in Neo4j: `ontology_demo_generic`, `ontology_demo_fibo`, `ontology_demo_fibo_plus_lawsuit`. Browse them at http://localhost:7474:

```cypher
MATCH (n)-[r]->(m) WHERE n._workspace_id = "ontology_demo_fibo_plus_lawsuit" RETURN n,r,m
```

**Now you try.** Pick a different FinDER doc by changing `DEMO_INDEX`, declare a class that fits its content, and re-run. The pattern is always: edit ontology → see new prompt → see new graph.